# ErrP Personalization — Ablation : Class-Weighted vs Uniform Loss

**Dataset**: INRIA BCI Challenge NER 2015 (26 subjects, LOSO evaluation)
**Goal**: Few-shot personalization of Error-Related Potential (ErrP) detection


## 1. Imports and Environment Setup

All required libraries loaded here. `pyriemann` installed silently if absent.
MNE logging suppressed. All warnings filtered.


In [ ]:
import numpy as np
import pandas as pd
import scipy
from scipy import signal as sp_signal, stats
from scipy.stats import wilcoxon
from scipy.linalg import sqrtm as mat_sqrt
from copy import deepcopy
import glob, re, os, sys, json, pickle, time, warnings, logging, random
from collections import defaultdict
from pathlib import Path
from typing import Dict, List, Tuple, Optional, Any

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim

from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (accuracy_score, balanced_accuracy_score,
                             f1_score, roc_auc_score, confusion_matrix)
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.linear_model import LogisticRegression

import mne
from mne.io import RawArray

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.auto import tqdm

logging.getLogger('mne').setLevel(logging.ERROR)
warnings.filterwarnings('ignore', category=FutureWarning)
warnings.filterwarnings('ignore', category=DeprecationWarning)

print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")


## 2. Configuration — Single Source of Truth

**All** hyperparameters live in `Config`. No execution cell overrides any field.
`Config.SFREQ`, `Config.N_CHANNELS`, `Config.N_TIMES`, and `Config.KERNEL_LENGTH`
are set at runtime after data loading (their values depend on the actual dataset).

Key design decisions encoded here:
- `SUBJECT_EMBED_DIM = 32` is used in SubjectEncoder, FiLM, and every call site  
- `K_SHOTS = [5, 10, 20]` — K=50 removed (not a few-shot regime)  
- `N_SEEDS = 3` — minimum for publishable variance estimates


In [ ]:
def _resolve_dataset_root(default="/kaggle/input/inria-bci-challenge/inria-bci-challenge"):
    candidates = [default]
    cwd = os.getcwd()
    candidates += [
        os.path.join(cwd, "Code", "inria-bci-challenge"),
        os.path.join(cwd, "inria-bci-challenge"),
        os.path.join(cwd, "..", "Code", "inria-bci-challenge"),
    ]
    seen, uniq = set(), []
    for p in candidates:
        n = os.path.normpath(p)
        if n not in seen:
            seen.add(n); uniq.append(n)
    for root in uniq:
        if (os.path.isfile(os.path.join(root, "TrainLabels.csv"))
                and os.path.isdir(os.path.join(root, "train"))):
            return root
    return os.path.normpath(default)


def _resolve_output_root(default="/kaggle/working/results_v2"):
    return default if os.path.isdir("/kaggle/working") else os.path.normpath(
        os.path.join(os.getcwd(), "Results", "results_v2"))


class Config:
    # ── Paths ──────────────────────────────────────────────────────────
    DATASET_ROOT  = _resolve_dataset_root()
    TRAIN_DIR     = os.path.join(DATASET_ROOT, "train")
    LABELS_FILE   = os.path.join(DATASET_ROOT, "TrainLabels.csv")

    OUTPUT_ROOT    = _resolve_output_root()
    RESULTS_DIR    = OUTPUT_ROOT
    FIGURES_DIR    = os.path.join(OUTPUT_ROOT, "figures")
    METRICS_DIR    = os.path.join(OUTPUT_ROOT, "metrics")
    CSV_DIR        = os.path.join(OUTPUT_ROOT, "csv")
    CHECKPOINT_DIR = os.path.join(OUTPUT_ROOT, "checkpoints")

    # ── Preprocessing ──────────────────────────────────────────────────
    TMIN             = -0.2      # epoch start relative to feedback onset (s)
    TMAX             =  0.6      # epoch end relative to feedback onset (s)
    BASELINE         = (-0.2, 0.0)
    LOWCUT           =  1.0      # bandpass lower cutoff (Hz)
    HIGHCUT          = 40.0      # bandpass upper cutoff (Hz)
    NOTCH_FREQ       = 50.0      # power-line notch (Hz)
    FILTER_ORDER     =  4        # Butterworth order
    ART_THRESHOLD_UV = 100.0     # artifact rejection threshold µV (peak-to-peak)

    # ── Feature extraction ─────────────────────────────────────────────
    PCA_COMPONENTS = 32

    # ── Experiment ─────────────────────────────────────────────────────
    K_SHOTS         = [5, 10, 20]   # K=50 removed (not few-shot)
    N_SEEDS         = 3
    RANDOM_SEEDS    = [42, 123, 456]
    N_EVAL_EPISODES = 10

    # ── Meta-learning ──────────────────────────────────────────────────
    N_META_ITERATIONS = 2000
    META_BATCH_SIZE   = 4
    N_SUPPORT         = 10
    N_QUERY           = 40
    INNER_LR          = 0.01
    OUTER_LR          = 5e-4
    INNER_STEPS       = 5

    # ── Encoder architecture ───────────────────────────────────────────
    ENCODER_HIDDEN  = 256   # 3-layer MLP
    ENCODER_HIDDEN2 = 128
    ENCODER_OUTPUT  = 64
    DROPOUT         = 0.3

    # ── Subject-conditioned meta-learner ───────────────────────────────
    SUBJECT_EMBED_DIM = 32   # consistent across SubjectEncoder and FiLM

    # ── EEGNet ─────────────────────────────────────────────────────────
    EEGNET_F1      = 8
    EEGNET_D       = 2
    EEGNET_F2      = 16
    EEGNET_DROPOUT = 0.25
    # KERNEL_LENGTH, SFREQ, N_CHANNELS, N_TIMES set at runtime after data load

    # ── Device ─────────────────────────────────────────────────────────
    DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')


for d in [Config.RESULTS_DIR, Config.FIGURES_DIR, Config.METRICS_DIR,
          Config.CSV_DIR, Config.CHECKPOINT_DIR]:
    os.makedirs(d, exist_ok=True)

print(f"Dataset root: {Config.DATASET_ROOT}")
print(f"Output root : {Config.OUTPUT_ROOT}")
print(f"Device      : {Config.DEVICE}")
print(f"K_SHOTS     : {Config.K_SHOTS}")
print(f"N_SEEDS     : {Config.N_SEEDS}")


## 3. Determinism and Reproducibility

`set_seed` enforces full determinism across Python random, NumPy, PyTorch CPU/GPU,
CUDA algorithm selection, and the CUBLAS workspace.

`freeze_batchnorm` prevents BatchNorm running statistics from updating during
inner-loop adaptation where batch sizes (K=5–20) are too small for reliable
batch statistics.


In [ ]:
def set_seed(seed: int = 42) -> None:
    """Enforce full determinism across all random sources."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.use_deterministic_algorithms(True, warn_only=True)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    os.environ['CUBLAS_WORKSPACE_CONFIG'] = ':4096:8'


def freeze_batchnorm(model: nn.Module) -> None:
    """Put all BatchNorm layers in eval mode and disable running stat updates.

    Essential during inner-loop adaptation: batch sizes of K=5-20 produce
    unreliable batch statistics that corrupt running mean/variance.
    """
    for m in model.modules():
        if isinstance(m, (nn.BatchNorm1d, nn.BatchNorm2d, nn.BatchNorm3d)):
            m.eval()
            m.track_running_stats = False


set_seed(Config.RANDOM_SEEDS[0])
print("Determinism enforced.")


## 4. Metrics — Primary: Balanced Accuracy and AUC-ROC

**Why NOT raw accuracy as primary metric:**

ErrP datasets have ~75% correct / ~25% error trials.
A classifier that always predicts "correct" achieves ~75% raw accuracy
while learning nothing at all.

Balanced accuracy (average recall per class) and AUC-ROC are immune to
class imbalance and are the standard in EEG BCI literature (Yasemin et al. 2023).

Raw accuracy is computed and reported as secondary reference only.


In [ ]:
def compute_comprehensive_metrics(
        predictions: np.ndarray,
        labels: np.ndarray,
        probabilities: Optional[np.ndarray] = None
) -> Dict[str, Any]:
    """Compute all metrics. Primary: balanced_accuracy, auroc. Secondary: accuracy.

    Args:
        predictions  : Predicted labels (0 or 1).
        labels       : Ground-truth labels (0 or 1).
        probabilities: Predicted probabilities shape (N,2) or (N,). Optional for AUC.

    Returns:
        Dict with balanced_accuracy, auroc, accuracy, f1_score, confusion matrix.
    """
    predictions = np.asarray(predictions).ravel()
    labels      = np.asarray(labels).ravel()

    metrics: Dict[str, Any] = {
        'accuracy'          : float(accuracy_score(labels, predictions)),
        'balanced_accuracy' : float(balanced_accuracy_score(labels, predictions)),
        'f1_score'          : float(f1_score(labels, predictions,
                                             average='binary', zero_division=0.0)),
    }

    cm = confusion_matrix(labels, predictions, labels=[0, 1])
    tn, fp, fn, tp = cm.ravel()
    metrics.update({'tn': int(tn), 'fp': int(fp), 'fn': int(fn), 'tp': int(tp),
                    'n_samples': len(labels),
                    'n_class_0': int(np.sum(labels == 0)),
                    'n_class_1': int(np.sum(labels == 1))})

    if probabilities is not None:
        probs = np.asarray(probabilities)
        p1 = probs[:, 1] if probs.ndim == 2 and probs.shape[1] == 2 else probs.ravel()
        try:
            metrics['auroc'] = float(roc_auc_score(labels, p1))
        except ValueError:
            metrics['auroc'] = float('nan')
    else:
        metrics['auroc'] = float('nan')

    return metrics


def compute_chance_level(labels: np.ndarray) -> Dict[str, float]:
    """Chance level = always predict majority class. Balanced acc = 0.5 by definition."""
    majority = int(np.bincount(labels).argmax())
    preds = np.full_like(labels, majority)
    return compute_comprehensive_metrics(preds, labels)


## 5. Results I/O — Checkpointing and CSV Export

After each LOSO fold, the per-subject result is saved to disk as JSON.
If training crashes at fold 20/26, re-running picks up from fold 21.

`save_per_subject_metrics` and `save_aggregate_metrics` produce CSVs
compatible with downstream statistical analysis.


In [ ]:
def _ckpt_path(output_dir: str, method: str, seed: int, subject: str) -> Path:
    p = Path(output_dir) / f"seed_{seed}" / "fold_checkpoints"
    p.mkdir(parents=True, exist_ok=True)
    return p / f"{method}_{subject}.json"


def _json_safe(obj):
    if isinstance(obj, (np.floating, float)):
        v = float(obj)
        return None if (v != v) else v   # NaN → None for JSON
    if isinstance(obj, (np.integer, int)):
        return int(obj)
    if isinstance(obj, np.ndarray):
        return obj.tolist()
    return obj


def save_subject_checkpoint(output_dir: str, method: str, seed: int,
                             subject: str, result: Dict) -> None:
    """Save single-subject result immediately after computation (crash-safe)."""
    path = _ckpt_path(output_dir, method, seed, subject)
    with open(path, 'w') as f:
        json.dump(result, f, default=_json_safe)


def load_subject_checkpoint(output_dir: str, method: str, seed: int,
                             subject: str) -> Optional[Dict]:
    """Load previously saved fold result, or None if not found."""
    path = _ckpt_path(output_dir, method, seed, subject)
    if path.exists():
        with open(path) as f:
            return json.load(f)
    return None


def save_per_subject_metrics(results: Dict, method_name: str, seed: int,
                              output_dir: str = Config.RESULTS_DIR) -> None:
    """Write per-subject × per-K metrics to CSV."""
    seed_dir = Path(output_dir) / f"seed_{seed}"
    seed_dir.mkdir(parents=True, exist_ok=True)
    rows = []
    for sid, sd in results.get('subjects', {}).items():
        for k, km in sd.get('k_shots', {}).items():
            rows.append({'method': method_name, 'seed': seed, 'subject_id': sid,
                         'k_shots': k,
                         **{m: km.get(m, float('nan'))
                            for m in ['accuracy', 'balanced_accuracy', 'f1_score', 'auroc',
                                      'n_samples', 'n_class_0', 'n_class_1',
                                      'tp', 'fp', 'tn', 'fn']}})
    pd.DataFrame(rows).to_csv(seed_dir / f"{method_name}_per_subject.csv", index=False)


def save_aggregate_metrics(results: Dict, method_name: str, seed: int,
                            k_shots: List[int],
                            output_dir: str = Config.RESULTS_DIR) -> None:
    """Write aggregate (mean ± std across subjects) metrics to CSV."""
    seed_dir = Path(output_dir) / f"seed_{seed}"
    seed_dir.mkdir(parents=True, exist_ok=True)
    rows = []
    for k in k_shots:
        vals = defaultdict(list)
        for sd in results.get('subjects', {}).values():
            km = sd.get('k_shots', {}).get(k, {})
            for m in ['accuracy', 'balanced_accuracy', 'f1_score', 'auroc']:
                v = km.get(m, float('nan'))
                if not (isinstance(v, float) and v != v):  # skip NaN
                    vals[m].append(v)
        row = {'method': method_name, 'seed': seed, 'k_shots': k}
        for m, vs in vals.items():
            row[f'{m}_mean'] = float(np.mean(vs)) if vs else float('nan')
            row[f'{m}_std']  = float(np.std(vs))  if vs else float('nan')
        rows.append(row)
    pd.DataFrame(rows).to_csv(seed_dir / f"{method_name}_aggregate.csv", index=False)


def normalize_results_dict(d: Dict[str, Dict]) -> Dict[str, Dict]:
    """Ensure consistent metric keys ('f1' → 'f1_score', fill missing with nan)."""
    out = {}
    for method, r in d.items():
        rc = deepcopy(r)
        for sd in rc.get('subjects', {}).values():
            for km in sd.get('k_shots', {}).values():
                if 'f1' in km and 'f1_score' not in km:
                    km['f1_score'] = km.pop('f1')
                for m in ['accuracy', 'balanced_accuracy', 'f1_score', 'auroc']:
                    km.setdefault(m, float('nan'))
        out[method] = rc
    return out


## 6. Statistical Tests with Effect Sizes

**Wilcoxon signed-rank test** on paired per-subject balanced accuracies.
Non-parametric, appropriate for N≈26 subjects, no normality assumption.

**Effect size**: rank-biserial correlation *r* reported alongside every p-value.
|r| < 0.1 = negligible, < 0.3 = small, < 0.5 = medium, ≥ 0.5 = large.

This follows the recommendation of Sullivan & Feinn (2012) that effect sizes
must accompany significance tests for clinical/BCI research.


In [ ]:
def paired_wilcoxon(results1: Dict, results2: Dict, k: int,
                    metric: str = 'balanced_accuracy') -> Dict:
    """Paired Wilcoxon signed-rank test at a given K, with rank-biserial effect size."""
    from scipy.stats import norm

    def _get(r, sid, k, m):
        v = r.get('subjects', {}).get(sid, {}).get('k_shots', {}).get(k, {}).get(m)
        return None if (v is None or (isinstance(v, float) and v != v)) else float(v)

    subs = sorted(set(results1.get('subjects', {})) & set(results2.get('subjects', {})))
    pairs = [(a, b) for s in subs
             for a, b in [(_get(results1, s, k, metric), _get(results2, s, k, metric))]
             if a is not None and b is not None]

    if len(pairs) < 2:
        return {'error': f'Only {len(pairs)} paired subjects', 'n': len(pairs)}

    a1 = np.array([p[0] for p in pairs])
    a2 = np.array([p[1] for p in pairs])
    diff = a1 - a2

    try:
        stat, p = wilcoxon(a1, a2, alternative='two-sided')
    except ValueError as e:
        return {'error': str(e), 'n': len(pairs)}

    n = len(pairs)
    z = norm.ppf(1 - p / 2) * np.sign(np.mean(diff))
    r = z / np.sqrt(n)
    interp = ('negligible' if abs(r) < 0.1
              else 'small' if abs(r) < 0.3
              else 'medium' if abs(r) < 0.5 else 'large')

    return {'metric': metric, 'k': k, 'n': n,
            'mean1': float(np.mean(a1)), 'mean2': float(np.mean(a2)),
            'mean_diff': float(np.mean(diff)), 'statistic': float(stat),
            'p_value': float(p), 'significant': bool(p < 0.05),
            'effect_r': float(r), 'effect_interp': interp}


def perform_statistical_tests(results_dict: Dict[str, Dict],
                               k_shots: List[int],
                               comparisons: List[Tuple[str, str]],
                               output_dir: str = Config.RESULTS_DIR,
                               seed: int = 42) -> pd.DataFrame:
    """Run all pairwise tests and save to CSV."""
    rows = []
    for m1, m2 in comparisons:
        if m1 not in results_dict or m2 not in results_dict:
            continue
        for k in k_shots:
            row = paired_wilcoxon(results_dict[m1], results_dict[m2], k)
            rows.append({'method1': m1, 'method2': m2, **row})

    df = pd.DataFrame(rows)
    out_dir = Path(output_dir) / f"seed_{seed}"
    out_dir.mkdir(parents=True, exist_ok=True)
    df.to_csv(out_dir / "statistical_tests.csv", index=False)
    return df


## 7. Dataset Loading

The INRIA BCI Challenge dataset: CSV files named `Data_S{N}_Sess{M}.csv`
for 26 subjects × 5 sessions. Labels from `TrainLabels.csv`.

`parse_label_id` extracts subject/session/feedback index from entries like
`S02_Sess01_FB013`. The feedback index is used for **index-based** epoch–label
alignment (not positional truncation, which is incorrect when any epoch is
dropped due to boundary effects).


In [ ]:
def parse_filename(filename: str) -> Tuple[Optional[str], Optional[int]]:
    """Extract (subject_id, session_num) from 'Data_S02_Sess01.csv'."""
    m = re.match(r'Data_S(\d+)_Sess(\d+)\.csv', filename)
    return (f"S{m.group(1)}", int(m.group(2))) if m else (None, None)


def index_dataset(data_dir: str) -> Dict[str, List[Tuple[int, str]]]:
    """Return {subject_id: [(session_num, filepath), ...]} sorted by session."""
    idx: Dict = defaultdict(list)
    for fp in glob.glob(os.path.join(data_dir, "*.csv")):
        sid, sess = parse_filename(os.path.basename(fp))
        if sid:
            idx[sid].append((sess, fp))
    for k in idx:
        idx[k].sort()
    return dict(idx)


def load_labels(labels_file: str) -> Dict[str, Dict[int, List[Tuple[int, int]]]]:
    """Parse TrainLabels.csv → {subject → {session → [(fb_idx, label)]}}."""
    df = pd.read_csv(labels_file)
    ld: Dict = defaultdict(lambda: defaultdict(list))
    for _, row in df.iterrows():
        m = re.match(r'S(\d+)_Sess(\d+)_FB(\d+)', str(row['IdFeedBack']))
        if not m:
            continue
        sid  = f"S{m.group(1)}"
        sess = int(m.group(2))
        fbi  = int(m.group(3))
        ld[sid][sess].append((fbi, int(row['Prediction'])))
    for sid in ld:
        for sess in ld[sid]:
            ld[sid][sess].sort(key=lambda x: x[0])
    return {k: dict(v) for k, v in ld.items()}


def build_dataset_mapping(train_index: Dict, labels_dict: Dict) -> Dict[str, List[Dict]]:
    """Join file index with label lists into a single mapping."""
    mapping = {}
    for sid, sessions in train_index.items():
        if sid not in labels_dict:
            continue
        mapping[sid] = []
        for sess, fp in sessions:
            if sess not in labels_dict[sid]:
                continue
            mapping[sid].append({'session_num': sess, 'filepath': fp,
                                  'labels': labels_dict[sid][sess]})
    return mapping


# ── Execute ──────────────────────────────────────────────────────────────
train_index     = index_dataset(Config.TRAIN_DIR)
labels_dict     = load_labels(Config.LABELS_FILE)
dataset_mapping = build_dataset_mapping(train_index, labels_dict)

print(f"Subjects found : {len(dataset_mapping)}")
for sid in sorted(dataset_mapping)[:3]:
    total = sum(len(s['labels']) for s in dataset_mapping[sid])
    print(f"  {sid}: {len(dataset_mapping[sid])} sessions, {total} labelled trials")


## 8. Preprocessing Pipeline

This is the literature-compliant pipeline following Yasemin et al. (2023),
Lawhern et al. (2018), and the INRIA BCI Challenge winning solution.

### Correct pipeline order

```
Load continuous CSV
  → Notch filter (50 Hz) on continuous signal
  → Bandpass filter (1–40 Hz) on continuous signal
  → Detect feedback event onsets (FeedBackEvent: 0→1 transitions)
  → Slice epochs (−200 ms to +600 ms)
  → Align epochs to labels by FEEDBACK INDEX (not positional truncation)
  → Artifact rejection: reject if peak-to-peak > 100 µV (on raw µV, before normalization)
  → Per-epoch per-channel baseline correction (subtract mean of [−200, 0] ms)
  → Concatenate sessions
```

**Why continuous filtering before epoching:**  
Filtering after epoching introduces spectral leakage at epoch boundaries  
(Gibb's phenomenon). The correct approach is always to filter the continuous  
signal first.

**Why artifact rejection before baseline correction:**  
Baseline correction subtracts the pre-stimulus mean from every time point.  
Artifact detection must operate on the raw µV signal to have a physiologically
meaningful threshold (100 µV is the standard in EEG literature).


In [ ]:
def load_continuous_eeg(filepath: str) -> Tuple[pd.DataFrame, float]:
    """Load CSV EEG file. Infer sampling frequency from Time column."""
    df = pd.read_csv(filepath)
    dt = float(np.median(np.diff(df['Time'].values[:200])))
    return df, 1.0 / dt


def get_eeg_channels(df: pd.DataFrame) -> List[str]:
    """Return column names excluding Time and FeedBackEvent."""
    return [c for c in df.columns if c not in ('Time', 'FeedBackEvent')]


def apply_filters_continuous(eeg_data: np.ndarray, sfreq: float,
                              lowcut: float, highcut: float,
                              notch_freq: float, order: int = 4) -> np.ndarray:
    """Apply notch then bandpass to continuous EEG (n_channels × n_samples).

    Uses zero-phase sosfiltfilt to avoid phase distortion.
    Applied BEFORE epoching to prevent boundary edge artifacts.

    Args:
        eeg_data  : Shape (n_channels, n_samples), raw µV.
        sfreq     : Sampling frequency in Hz.
        lowcut    : Bandpass lower cutoff (Hz).
        highcut   : Bandpass upper cutoff (Hz).
        notch_freq: Power-line notch (Hz).
        order     : Butterworth filter order.
    """
    nyq = sfreq / 2.0
    out = eeg_data.copy()

    # 1. Notch filter
    nl, nh = (notch_freq - 1.0) / nyq, (notch_freq + 1.0) / nyq
    if 0 < nl < 1 and 0 < nh < 1:
        sos = sp_signal.butter(order, [nl, nh], btype='bandstop', output='sos')
        out = sp_signal.sosfiltfilt(sos, out, axis=1)

    # 2. Bandpass filter
    lo, hi = lowcut / nyq, min(highcut / nyq, 0.9999)
    if 0 < lo < hi < 1:
        sos = sp_signal.butter(order, [lo, hi], btype='band', output='sos')
        out = sp_signal.sosfiltfilt(sos, out, axis=1)

    return out


def detect_feedback_events(df: pd.DataFrame) -> np.ndarray:
    """Return sample indices of 0→1 transitions in FeedBackEvent column."""
    ev = df['FeedBackEvent'].values
    return np.where(np.diff(np.concatenate([[0], ev])) == 1)[0]


def create_epochs(eeg_data: np.ndarray, event_indices: np.ndarray,
                  sfreq: float, tmin: float, tmax: float
                  ) -> Tuple[np.ndarray, np.ndarray, List[int]]:
    """Slice epochs from filtered continuous EEG.

    Returns:
        epochs_data     : (n_valid, n_ch, n_times)
        times           : (n_times,) time axis in seconds
        valid_positions : list mapping epoch index → original event position
    """
    n_before = int(abs(tmin) * sfreq)
    n_after  = int(tmax * sfreq)
    n_total  = eeg_data.shape[1]
    epochs, valid_pos = [], []
    for pos, idx in enumerate(event_indices):
        s, e = idx - n_before, idx + n_after
        if s >= 0 and e <= n_total:
            epochs.append(eeg_data[:, s:e])
            valid_pos.append(pos)
    arr   = np.array(epochs) if epochs else np.empty((0, eeg_data.shape[0], n_before + n_after))
    times = np.linspace(tmin, tmax, n_before + n_after, endpoint=False)
    return arr, times, valid_pos


def align_epochs_with_labels_by_index(
        epochs_data: np.ndarray,
        valid_positions: List[int],
        session_labels: List[Tuple[int, int]]
) -> Tuple[np.ndarray, np.ndarray]:
    """Align epochs to labels using the feedback index — NOT positional truncation.

    Positional truncation (min(n_epochs, n_labels)) is WRONG:
    if any epoch is dropped at a boundary, all subsequent labels shift silently.

    This function matches epoch i to the label whose event position equals
    valid_positions[i], using the ordered feedback list as the position reference.
    """
    pos_to_label = {pos: lbl for pos, (_, lbl) in enumerate(session_labels)}
    aligned_e, aligned_l = [], []
    for ep_i, orig_pos in enumerate(valid_positions):
        if orig_pos in pos_to_label:
            aligned_e.append(epochs_data[ep_i])
            aligned_l.append(pos_to_label[orig_pos])
    if not aligned_e:
        return np.empty((0,) + epochs_data.shape[1:]), np.empty(0, dtype=int)
    return np.array(aligned_e), np.array(aligned_l, dtype=int)


def reject_artifacts(epochs_data: np.ndarray,
                     threshold_uv: float = 100.0) -> np.ndarray:
    """Return boolean mask (True = keep) for epochs below peak-to-peak threshold.

    Applied on RAW µV BEFORE baseline correction or z-score.
    The 100 µV threshold is standard in ERP literature (Luck 2014).
    """
    ptp = np.ptp(epochs_data, axis=2).max(axis=1)   # max ptp across channels
    return ptp <= threshold_uv


def baseline_correct(epochs_data: np.ndarray, times: np.ndarray,
                     baseline: Tuple[float, float] = (-0.2, 0.0)) -> np.ndarray:
    """Subtract per-epoch per-channel mean of the baseline window.

    Applied AFTER artifact rejection on CLEAN epochs.
    Computed independently for each epoch and each channel.
    """
    t0, t1 = baseline
    mask = (times >= t0) & (times <= t1)
    assert mask.sum() > 0, f"Baseline window {baseline} is empty (times range: {times[[0,-1]]})"
    bl_mean = epochs_data[:, :, mask].mean(axis=2, keepdims=True)  # (N, C, 1)
    return epochs_data - bl_mean


def process_subject_data(subject_id: str, session_infos: List[Dict]) -> Optional[Dict]:
    """Complete preprocessing pipeline for one subject across all sessions.

    Pipeline:
        continuous load → continuous filter → epoch → index-align →
        artifact reject (µV) → baseline correct → concatenate sessions

    Returns dict: epochs (N,C,T), labels (N,), ch_names, times, sfreq,
                  n_rejected, n_raw.
    """
    all_epochs, all_labels = [], []
    sfreqs: List[float] = []
    ch_names_ref: Optional[List[str]] = None
    times_ref: Optional[np.ndarray] = None
    total_raw = total_rejected = 0

    for sess_info in session_infos:
        try:
            df, sfreq = load_continuous_eeg(sess_info['filepath'])
            ch_names  = get_eeg_channels(df)
            raw_data  = df[ch_names].values.T  # (n_ch, n_samples)

            # Step 1: filter continuous signal
            raw_filt = apply_filters_continuous(
                raw_data, sfreq,
                Config.LOWCUT, Config.HIGHCUT, Config.NOTCH_FREQ, Config.FILTER_ORDER)

            # Step 2: detect events + create epochs
            events = detect_feedback_events(df)
            epo, times, valid_pos = create_epochs(
                raw_filt, events, sfreq, Config.TMIN, Config.TMAX)
            if len(epo) == 0:
                continue

            # Step 3: align by feedback index
            epo_al, lbl_al = align_epochs_with_labels_by_index(
                epo, valid_pos, sess_info['labels'])
            if len(epo_al) == 0:
                continue

            # Step 4: artifact rejection on raw µV
            mask = reject_artifacts(epo_al, Config.ART_THRESHOLD_UV)
            n_rej = int((~mask).sum())
            total_raw      += len(epo_al)
            total_rejected += n_rej
            epo_clean  = epo_al[mask]
            lbl_clean  = lbl_al[mask]
            if len(epo_clean) == 0:
                continue

            # Step 5: baseline correction on clean epochs
            epo_bc = baseline_correct(epo_clean, times, Config.BASELINE)

            all_epochs.append(epo_bc)
            all_labels.append(lbl_clean)
            sfreqs.append(sfreq)
            ch_names_ref = ch_names
            times_ref    = times

        except Exception as exc:
            print(f"  [{subject_id}] session {sess_info.get('session_num','?')} failed: {exc}")

    if not all_epochs:
        return None

    epochs_cat = np.concatenate(all_epochs, axis=0).astype(np.float32)
    labels_cat = np.concatenate(all_labels, axis=0).astype(int)
    sfreq_mean = float(np.mean(sfreqs))
    rej_rate   = 100.0 * total_rejected / max(total_raw, 1)
    cc = np.bincount(labels_cat, minlength=2)
    print(f"  {subject_id}: {len(epochs_cat)} epochs "
          f"(rej {total_rejected}/{total_raw}, {rej_rate:.1f}%) "
          f"| cls0={cc[0]} cls1={cc[1]}")

    return {'epochs': epochs_cat, 'labels': labels_cat,
            'ch_names': ch_names_ref, 'times': times_ref,
            'sfreq': sfreq_mean, 'n_sessions': len(session_infos),
            'n_rejected': total_rejected, 'n_raw': total_raw}


# ── Run preprocessing ────────────────────────────────────────────────────
print("Preprocessing all subjects (filter → epoch → align → reject → baseline)...")
preprocessed_subjects_data: Dict[str, Dict] = {}
failed_subjects: List[str] = []

for sid in tqdm(sorted(dataset_mapping.keys()), desc="Preprocessing"):
    result = process_subject_data(sid, dataset_mapping[sid])
    if result is not None:
        preprocessed_subjects_data[sid] = result
    else:
        failed_subjects.append(sid)
        print(f"  FAILED: {sid}")

print(f"\nSucceeded: {len(preprocessed_subjects_data)} subjects")
if failed_subjects:
    print(f"Failed   : {failed_subjects}")

# Set runtime Config fields
first_subj = list(preprocessed_subjects_data.values())[0]
Config.SFREQ         = first_subj['sfreq']
Config.N_CHANNELS    = len(first_subj['ch_names'])
Config.N_TIMES       = first_subj['epochs'].shape[2]
Config.KERNEL_LENGTH = int(Config.SFREQ // 2)

print(f"\nSignal:  sfreq={Config.SFREQ:.1f} Hz  |  n_ch={Config.N_CHANNELS}"
      f"  |  n_times={Config.N_TIMES}  |  kernel_length={Config.KERNEL_LENGTH}")


## 9. Feature Extraction and LOSO-Isolated PCA

**Two processing branches:**

**Branch A — Classical ML** (MLP encoders, MAML, ProtoNets, Matching, Reptile)  
Features: flattened temporal waveform (n_channels × n_times) → PCA(32).  
Temporal features outperform bandpower for ERP detection because discriminative
information is carried by waveform morphology (Ne/ERN at ~250 ms, Pe at ~320 ms),
not spectral power (Yasemin et al. 2023).

**Branch B — Deep / Riemannian** (EEGNet, Riemannian baselines)  
Raw filtered waveforms (n_channels × n_times). EEGNet learns temporal and
spatial filters end-to-end. Riemannian methods use covariance matrices.

### LOSO PCA isolation
PCA is fitted on the N-1 training subjects' features only.
Test subject features are transformed using the training PCA.
No test subject statistics contaminate the PCA transformation.


In [ ]:
def extract_temporal_features(epochs: np.ndarray) -> np.ndarray:
    """Flatten (N, C, T) → (N, C*T). Temporal waveform features for meta-learning."""
    n, c, t = epochs.shape
    return epochs.reshape(n, c * t)


def extract_features_all_subjects(ppd: Dict[str, Dict]) -> Dict[str, Dict]:
    """Extract and return temporal features for all subjects."""
    out = {}
    for sid, data in tqdm(sorted(ppd.items()), desc="Feature extraction"):
        out[sid] = {'subject_id': sid,
                    'features': extract_temporal_features(data['epochs']).astype(np.float32),
                    'labels':   data['labels']}
    return out


class PCAReducer:
    """Fit StandardScaler + whitened PCA on training data; transform any data."""
    def __init__(self, n_components: int = 32, seed: int = 42):
        self.scaler    = StandardScaler()
        self.pca       = PCA(n_components=n_components, whiten=True, random_state=seed)
        self.is_fitted = False

    def fit(self, X: np.ndarray) -> 'PCAReducer':
        self.pca.fit(self.scaler.fit_transform(X))
        self.is_fitted = True
        return self

    def transform(self, X: np.ndarray) -> np.ndarray:
        return self.pca.transform(self.scaler.transform(X)).astype(np.float32)


def apply_pca_loso(subjects_features: Dict[str, Dict],
                   n_components: int = Config.PCA_COMPONENTS) -> Dict[str, Dict]:
    """Build LOSO PCA splits. Each fold's PCA fitted on N-1 training subjects.

    Each fold contains:
      test            : {features (PCA-reduced), labels}
      pca             : fitted PCAReducer for this fold
      train_subjects  : list of training subject IDs
      subjects_features: reference to raw feature dict (for training transforms)
    """
    sids = sorted(subjects_features.keys())
    splits = {}
    for test_sid in tqdm(sids, desc="LOSO PCA"):
        train_sids = [s for s in sids if s != test_sid]
        train_X    = np.concatenate([subjects_features[s]['features'] for s in train_sids])
        reducer    = PCAReducer(n_components=n_components).fit(train_X)
        splits[test_sid] = {
            'test'             : {'features': reducer.transform(subjects_features[test_sid]['features']),
                                  'labels'  : subjects_features[test_sid]['labels']},
            'pca'              : reducer,
            'train_subjects'   : train_sids,
            'subjects_features': subjects_features,
        }
    return splits


# ── Execute ──────────────────────────────────────────────────────────────
subjects_features = extract_features_all_subjects(preprocessed_subjects_data)
loso_splits       = apply_pca_loso(subjects_features, Config.PCA_COMPONENTS)

first_fold = loso_splits[sorted(loso_splits)[0]]
INPUT_DIM  = first_fold['test']['features'].shape[1]
print(f"PCA output dim : {INPUT_DIM}")
print(f"LOSO folds     : {len(loso_splits)}")
print(f"Train subjects per fold: {len(first_fold['train_subjects'])}")


In [ ]:
class EEGEncoder(nn.Module):
    """
    Meta-learned EEG feature encoder.

    Architecture: input → 256 → 128 → 64 (representation)
    Each hidden layer: Linear → LayerNorm → GELU → Dropout(0.3)
    Final layer:       Linear  [NO activation, NO dropout — unconstrained embedding]

    Rationale:
    - LayerNorm: makes activations amplitude-invariant across subjects
    - GELU: better gradient flow than ReLU for meta-learning outer loops
    - No final activation: unconstrained embedding (ReLU would discard ~50% of space)
    - Size 256→128→64: capacity without over-parameterization at 32-dim PCA input
    """
    def __init__(self, input_dim: int,
                 h1: int = 256, h2: int = 128, out_dim: int = 64,
                 dropout: float = 0.3):
        super().__init__()
        self.fc1 = nn.Linear(input_dim, h1)
        self.ln1 = nn.LayerNorm(h1)
        self.dp1 = nn.Dropout(dropout)
        self.fc2 = nn.Linear(h1, h2)
        self.ln2 = nn.LayerNorm(h2)
        self.dp2 = nn.Dropout(dropout)
        self.fc3 = nn.Linear(h2, out_dim)    # representation layer

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.dp1(F.gelu(self.ln1(self.fc1(x))))
        x = self.dp2(F.gelu(self.ln2(self.fc2(x))))
        return self.fc3(x)   # no activation


class SimpleTaskHead(nn.Module):
    """Linear classifier on encoder representation. Adapted per-subject in inner loop."""
    def __init__(self, input_dim: int = 64, num_classes: int = 2):
        super().__init__()
        self.fc = nn.Linear(input_dim, num_classes)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.fc(x)


class MetaEEGClassifier(nn.Module):
    """Full model: EEGEncoder + SimpleTaskHead. Used by MAML, ANIL, Reptile."""
    def __init__(self, input_dim: int,
                 h1: int = 256, h2: int = 128, enc_out: int = 64,
                 dropout: float = 0.3, num_classes: int = 2):
        super().__init__()
        self.encoder   = EEGEncoder(input_dim, h1, h2, enc_out, dropout)
        self.task_head = SimpleTaskHead(enc_out, num_classes)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.task_head(self.encoder(x))

    def get_repr(self, x: torch.Tensor) -> torch.Tensor:
        """Encoder representation without classification."""
        return self.encoder(x)


# ── Sanity check ─────────────────────────────────────────────────────────
_m = MetaEEGClassifier(INPUT_DIM)
_x = torch.randn(4, INPUT_DIM)
_y = _m(_x)
assert _y.shape == (4, 2), f"Unexpected output shape: {_y.shape}"
n_params = sum(p.numel() for p in _m.parameters())
print(f"MetaEEGClassifier: {n_params:,} parameters")
del _m, _x, _y


## 13. MAML and ANIL

### Full-MAML vs ANIL

- **Full-MAML** (`freeze_encoder=False`): both encoder AND task head adapt in inner loop.  
  More expressive but requires Hessian computation (or first-order approximation).  
- **ANIL** (`freeze_encoder=True`): only task head adapts; encoder is frozen inner-loop.  
  Raghu et al. (2020) showed ANIL matches MAML quality at much lower compute cost.

### Functional inner loop

Uses a shallow parameter dict copy (NOT `.clone()`) so the query loss gradient
flows back through the inner-loop update to the original meta-model parameters.
Cloning would stop gradients at the inner loop and break outer-loop learning.


In [ ]:
class MAML_Encoder:
    """
    MAML / ANIL for EEG-based BCI personalization.

    Outer: Adam + CosineAnnealingLR + gradient clipping (max_norm=1.0)
    Inner: functional gradient updates (correct gradient flow)
    Loss : class-weighted cross-entropy (handles 3:1 ErrP imbalance)
    Episodes: balanced (K/2 per class support and query)
    """
    def __init__(self, input_dim: int,
                 h1: int = Config.ENCODER_HIDDEN,
                 h2: int = Config.ENCODER_HIDDEN2,
                 enc_out: int = Config.ENCODER_OUTPUT,
                 num_classes: int = 2,
                 inner_lr: float = Config.INNER_LR,
                 outer_lr: float = Config.OUTER_LR,
                 inner_steps: int = Config.INNER_STEPS,
                 n_meta_iterations: int = Config.N_META_ITERATIONS,
                 freeze_encoder_inner: bool = False,
                 first_order: bool = True,
                 use_weighting: bool = True,          # ← ABLATION 4 flag
                 device: str = str(Config.DEVICE)):
        self.inner_lr    = inner_lr
        self.inner_steps = inner_steps
        self.freeze_enc  = freeze_encoder_inner
        self.first_order = first_order
        self.use_weighting = use_weighting             # ← ABLATION 4 flag
        self.device      = torch.device(device)

        self.meta_model = MetaEEGClassifier(
            input_dim, h1, h2, enc_out, Config.DROPOUT, num_classes).to(self.device)
        self.meta_opt   = optim.Adam(self.meta_model.parameters(), lr=outer_lr)
        self.scheduler  = optim.lr_scheduler.CosineAnnealingLR(
            self.meta_opt, T_max=n_meta_iterations, eta_min=outer_lr * 0.01)

    # ── Utility: balanced episode sampling ────────────────────────────────
    @staticmethod
    def _balanced_episode(features: np.ndarray, labels: np.ndarray,
                           n_support: int, n_query: int
                           ) -> Tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray]:
        """Sample a balanced episode with n_support/2 + n_query/2 per class."""
        unique = np.unique(labels)
        sp_per = max(1, n_support // len(unique))
        qp_per = max(1, n_query   // len(unique)) if n_query > 0 else 0
        sxl, syl, qxl, qyl = [], [], [], []
        for cls in unique:
            idx = np.where(labels == cls)[0]
            n   = sp_per + qp_per
            if len(idx) < n:
                sp = max(1, len(idx) // 2); qp = len(idx) - sp
            else:
                sp, qp = sp_per, qp_per
            perm = np.random.permutation(idx)
            sxl.append(features[perm[:sp]]);    syl.append(labels[perm[:sp]])
            qxl.append(features[perm[sp:sp+qp]]); qyl.append(labels[perm[sp:sp+qp]])
        return (np.concatenate(sxl), np.concatenate(syl),
                np.concatenate(qxl), np.concatenate(qyl))

    # ── Utility: class weights ────────────────────────────────────────────
    @staticmethod
    def _class_weights(labels: np.ndarray, device: torch.device,
                       n_classes: int = 2) -> torch.Tensor:
        counts = np.bincount(labels, minlength=n_classes).astype(float)
        counts = np.where(counts == 0, 1.0, counts)
        w = 1.0 / counts; w = w / w.sum() * n_classes
        return torch.FloatTensor(w).to(device)

    # ── ABLATION 4: weight selector ───────────────────────────────────────
    def _cw(self, labels: np.ndarray) -> torch.Tensor:
        """Return class weights or uniform weights depending on use_weighting flag.

        use_weighting=True  (default / primary):  inverse-frequency weights
        use_weighting=False (ablation):            uniform weights [0.5, 0.5]

        Called in meta_update (support + query) and adapt_and_evaluate (support).
        Replaces all direct calls to _class_weights throughout the class.
        """
        if self.use_weighting:
            return self._class_weights(labels, self.device)
        else:
            return torch.FloatTensor([1.0, 1.0]).to(self.device)

    # ── Functional forward pass ──────────────────────────────────────────
    def _functional_forward(self, x: torch.Tensor, params: dict) -> torch.Tensor:
        """Forward pass using parameter dict (enables functional gradient flow)."""
        h = F.dropout(F.gelu(F.layer_norm(
                F.linear(x, params['encoder.fc1.weight'], params['encoder.fc1.bias']),
                [params['encoder.fc1.weight'].shape[0]])),
                p=Config.DROPOUT, training=self.meta_model.training)
        h = F.dropout(F.gelu(F.layer_norm(
                F.linear(h, params['encoder.fc2.weight'], params['encoder.fc2.bias']),
                [params['encoder.fc2.weight'].shape[0]])),
                p=Config.DROPOUT, training=self.meta_model.training)
        h = F.linear(h, params['encoder.fc3.weight'], params['encoder.fc3.bias'])
        return F.linear(h, params['task_head.fc.weight'], params['task_head.fc.bias'])

    # ── Functional inner loop ─────────────────────────────────────────────
    def _inner_loop(self, sx: torch.Tensor, sy: torch.Tensor,
                    params: dict, cw: torch.Tensor) -> Tuple[dict, float]:
        """Inner-loop gradient adaptation using functional gradients.

        Uses SHALLOW dict copy — tensors are shared, not cloned.
        This preserves the computational graph from query loss back to meta-params.
        """
        adapted   = dict(params)   # shallow copy — DO NOT use deepcopy here
        to_update = ({k: v for k, v in adapted.items() if 'task_head' in k}
                     if self.freeze_enc else adapted)
        total_loss = 0.0
        for _ in range(self.inner_steps):
            logits = self._functional_forward(sx, adapted)
            loss   = F.cross_entropy(logits, sy, weight=cw)
            grads  = torch.autograd.grad(
                loss, to_update.values(),
                create_graph=not self.first_order, allow_unused=True)
            for (name, _), g in zip(list(to_update.items()), grads):
                if g is not None:
                    g_ = g.detach() if self.first_order else g
                    adapted[name] = adapted[name] - self.inner_lr * g_
            to_update = {k: adapted[k] for k in to_update}
            total_loss += loss.item()
        return adapted, total_loss / self.inner_steps

    def meta_update(self, tasks: list) -> dict:
        """Outer-loop meta-update across a batch of tasks."""
        self.meta_opt.zero_grad()
        params = {n: p for n, p in self.meta_model.named_parameters()}
        meta_losses, q_accs = [], []

        for sx, sy, qx, qy in tasks:
            sx = torch.FloatTensor(sx).to(self.device)
            sy = torch.LongTensor(sy).to(self.device)
            qx = torch.FloatTensor(qx).to(self.device)
            qy = torch.LongTensor(qy).to(self.device)
            cw   = self._cw(sy.cpu().numpy())              # ← was _class_weights

            adapted, _ = self._inner_loop(sx, sy, params, cw)
            q_logits   = self._functional_forward(qx, adapted)
            q_cw       = self._cw(qy.cpu().numpy())        # ← was _class_weights
            q_loss     = F.cross_entropy(q_logits, qy, weight=q_cw)
            meta_losses.append(q_loss)
            with torch.no_grad():
                q_accs.append((q_logits.argmax(1) == qy).float().mean().item())

        meta_loss = torch.stack(meta_losses).mean()
        meta_loss.backward()
        nn.utils.clip_grad_norm_(self.meta_model.parameters(), max_norm=1.0)
        self.meta_opt.step()
        self.scheduler.step()
        return {'meta_loss': meta_loss.item(), 'query_acc': float(np.mean(q_accs))}

    def adapt_and_evaluate(self, support_X: np.ndarray, support_y: np.ndarray,
                            query_X: np.ndarray, query_y: np.ndarray) -> dict:
        """Test-time adaptation and evaluation."""
        adapted_model = deepcopy(self.meta_model)
        freeze_batchnorm(adapted_model)

        if self.freeze_enc:
            adapt_params = list(adapted_model.task_head.parameters())
            for p in adapted_model.encoder.parameters():
                p.requires_grad_(False)
        else:
            adapt_params = list(adapted_model.parameters())

        sx  = torch.FloatTensor(support_X).to(self.device)
        sy  = torch.LongTensor(support_y).to(self.device)
        cw  = self._cw(support_y)                          # ← was _class_weights
        opt = optim.SGD(adapt_params, lr=self.inner_lr)

        adapted_model.train()
        for _ in range(self.inner_steps):
            opt.zero_grad()
            F.cross_entropy(adapted_model(sx), sy, weight=cw).backward()
            opt.step()

        adapted_model.eval()
        with torch.no_grad():
            lgts  = adapted_model(torch.FloatTensor(query_X).to(self.device))
            probs = F.softmax(lgts, 1).cpu().numpy()
            preds = lgts.argmax(1).cpu().numpy()
        del adapted_model
        return compute_comprehensive_metrics(preds, query_y, probs)


def train_maml_loso(
        loso_splits: dict, k_shots: list,
        freeze_encoder_inner: bool = False,
        n_meta_iterations: int = Config.N_META_ITERATIONS,
        meta_batch_size: int = Config.META_BATCH_SIZE,
        n_support: int = Config.N_SUPPORT, n_query: int = Config.N_QUERY,
        inner_lr: float = Config.INNER_LR, outer_lr: float = Config.OUTER_LR,
        inner_steps: int = Config.INNER_STEPS,
        use_weighting: bool = True,                        # ← ABLATION 4 flag
        device: str = str(Config.DEVICE), seed: int = 42,
        output_dir: str = Config.RESULTS_DIR,
        method_name: str = 'MAML') -> dict:
    """LOSO MAML/ANIL training with checkpointing and zero-shot evaluation."""
    set_seed(seed)
    all_results: dict = {}

    for fold_idx, (test_sid, fold) in enumerate(
            tqdm(loso_splits.items(), desc=f"{method_name} LOSO")):
        fold_seed = seed + fold_idx * 13
        set_seed(fold_seed)

        ckpt = load_subject_checkpoint(output_dir, method_name, seed, test_sid)
        if ckpt is not None:
            all_results[test_sid] = ckpt
            continue

        train_sids = fold['train_subjects']
        train_dict = {s: {'features': fold['pca'].transform(
                              fold['subjects_features'][s]['features']),
                          'labels'  : fold['subjects_features'][s]['labels']}
                      for s in train_sids}
        input_dim = fold['test']['features'].shape[1]

        agent = MAML_Encoder(input_dim=input_dim, freeze_encoder_inner=freeze_encoder_inner,
                             inner_lr=inner_lr, outer_lr=outer_lr, inner_steps=inner_steps,
                             n_meta_iterations=n_meta_iterations,
                             first_order=freeze_encoder_inner,
                             use_weighting=use_weighting,  # ← ABLATION 4 flag
                             device=device)

        # ── Meta-training ─────────────────────────────────────────────────
        for it in range(n_meta_iterations):
            batch = np.random.choice(train_sids,
                                     size=min(meta_batch_size, len(train_sids)),
                                     replace=False)
            tasks = []
            for s in batch:
                f, l = train_dict[s]['features'], train_dict[s]['labels']
                if len(f) < n_support + n_query:
                    continue
                sx, sy, qx, qy = MAML_Encoder._balanced_episode(f, l, n_support, n_query)
                tasks.append((sx, sy, qx, qy))
            if tasks:
                agent.meta_update(tasks)

        # ── Zero-shot baseline ─────────────────────────────────────────────
        test_X = fold['test']['features']
        test_y = fold['test']['labels']
        agent.meta_model.eval()
        with torch.no_grad():
            lgts = agent.meta_model(torch.FloatTensor(test_X).to(agent.device))
            zs_probs = F.softmax(lgts, 1).cpu().numpy()
            zs_preds = lgts.argmax(1).cpu().numpy()
        zero_shot = compute_comprehensive_metrics(zs_preds, test_y, zs_probs)

        # ── K-shot evaluation ──────────────────────────────────────────────
        subject_results = {'subject_id': test_sid,
                           'zero_shot': zero_shot, 'k_shots': {}}
        for k in k_shots:
            eps = []
            for ep in range(Config.N_EVAL_EPISODES):
                set_seed(fold_seed + k * 100 + ep)
                sx, sy, qx, qy = MAML_Encoder._balanced_episode(
                    test_X, test_y, k, min(len(test_y) - k, 200))
                if len(qy) == 0:
                    continue
                eps.append(agent.adapt_and_evaluate(sx, sy, qx, qy))
            agg = {}
            for m in ['accuracy', 'balanced_accuracy', 'f1_score', 'auroc']:
                vals = [e[m] for e in eps if not (e[m] != e[m])]
                agg[m] = float(np.mean(vals)) if vals else float('nan')
            subject_results['k_shots'][k] = agg

        all_results[test_sid] = subject_results
        save_subject_checkpoint(output_dir, method_name, seed, test_sid, subject_results)
        del agent
        if 'cuda' in str(device):
            torch.cuda.empty_cache()

    return {'method': method_name, 'subjects': all_results,
            'k_shots': k_shots, 'seed': seed}

## 20. LOSO Isolation Verification

Verifies that no test subject's data appears in any corresponding training set
and that PCA was fitted correctly on training subjects only.

**This cell must pass cleanly before any training begins.**
A contaminated PCA would inflate test performance unrealistically.


In [ ]:
def verify_loso_isolation(loso_splits: dict, verbose: bool = True) -> bool:
    """Comprehensive LOSO isolation check.

    Verifies:
    (a) Test subject not in own training set
    (b) PCA fitted and output dimension correct
    (c) No NaN or Inf in test features
    """
    all_ok = True
    for test_sid, fold in loso_splits.items():
        # (a)
        if test_sid in fold['train_subjects']:
            print(f"FAIL: {test_sid} in own training set!")
            all_ok = False

        # (b)
        tdim = fold['test']['features'].shape[1]
        pdim = fold['pca'].pca.n_components_
        if tdim != pdim:
            print(f"FAIL: {test_sid} dim mismatch {tdim} != {pdim}")
            all_ok = False
        if not fold['pca'].is_fitted:
            print(f"FAIL: {test_sid} PCA not fitted!")
            all_ok = False

        # (c)
        if np.any(~np.isfinite(fold['test']['features'])):
            print(f"FAIL: {test_sid} contains NaN/Inf in features!")
            all_ok = False

    if all_ok and verbose:
        print(f"OK  LOSO isolation verified: {len(loso_splits)} folds, all clean.")

    # Also check support/query disjointness
    first_sid = sorted(loso_splits.keys())[0]
    test_X = loso_splits[first_sid]['test']['features']
    test_y = loso_splits[first_sid]['test']['labels']
    for k in Config.K_SHOTS:
        sx, sy, qx, qy = MAML_Encoder._balanced_episode(test_X, test_y, k, 40)
        sx_set = {tuple(x.round(6)) for x in sx}
        qx_set = {tuple(x.round(6)) for x in qx}
        assert not (sx_set & qx_set), f"K={k}: support/query overlap!"
    if verbose:
        print(f"OK  Support/query disjointness verified for K={Config.K_SHOTS}")

    return all_ok


verify_loso_isolation(loso_splits)


## 21. Execution Pipeline

All methods run in sequence across `N_SEEDS = 3` independent seeds.

After each LOSO fold, results are checkpointed to disk — interrupted runs
continue from the last completed fold on restart.


In [ ]:
DEVICE     = str(Config.DEVICE)
K_SHOTS    = Config.K_SHOTS
SEEDS      = Config.RANDOM_SEEDS
OUTPUT_DIR = Config.RESULTS_DIR

# Ablation 4: Class-Weighted vs Uniform Loss
# Four variants: Full-MAML and MAML-ANIL, each with weighted and uniform loss
# use_weighting=True  → primary (existing results, can load from checkpoint)
# use_weighting=False → ablation (new runs)

all_results_by_seed: Dict[int, Dict[str, dict]] = {}

for current_seed in SEEDS:
    print(f"\n{'='*60}\nSEED {current_seed}\n{'='*60}")
    set_seed(current_seed)
    seed_results: Dict[str, dict] = {}

    # ── Full-MAML weighted (primary — loads from checkpoint if already run) ──
    seed_results['Full-MAML'] = train_maml_loso(
        loso_splits, K_SHOTS, freeze_encoder_inner=False,
        use_weighting=True, device=DEVICE,
        seed=current_seed, output_dir=OUTPUT_DIR,
        method_name='Full-MAML')
    save_per_subject_metrics(seed_results['Full-MAML'],
                             'Full-MAML', current_seed, OUTPUT_DIR)
    save_aggregate_metrics(seed_results['Full-MAML'],
                           'Full-MAML', current_seed, K_SHOTS, OUTPUT_DIR)
    print("  Full-MAML (weighted) done")

    # ── Full-MAML uniform (ablation) ──────────────────────────────────────
    seed_results['Full-MAML-Uniform'] = train_maml_loso(
        loso_splits, K_SHOTS, freeze_encoder_inner=False,
        use_weighting=False, device=DEVICE,
        seed=current_seed, output_dir=OUTPUT_DIR,
        method_name='Full-MAML-Uniform')
    save_per_subject_metrics(seed_results['Full-MAML-Uniform'],
                             'Full-MAML-Uniform', current_seed, OUTPUT_DIR)
    save_aggregate_metrics(seed_results['Full-MAML-Uniform'],
                           'Full-MAML-Uniform', current_seed, K_SHOTS, OUTPUT_DIR)
    print("  Full-MAML-Uniform done")

    # ── MAML-ANIL weighted (primary — loads from checkpoint if already run) ─
    seed_results['MAML-ANIL'] = train_maml_loso(
        loso_splits, K_SHOTS, freeze_encoder_inner=True,
        use_weighting=True, device=DEVICE,
        seed=current_seed, output_dir=OUTPUT_DIR,
        method_name='MAML-ANIL')
    save_per_subject_metrics(seed_results['MAML-ANIL'],
                             'MAML-ANIL', current_seed, OUTPUT_DIR)
    save_aggregate_metrics(seed_results['MAML-ANIL'],
                           'MAML-ANIL', current_seed, K_SHOTS, OUTPUT_DIR)
    print("  MAML-ANIL (weighted) done")

    # ── MAML-ANIL uniform (ablation) ──────────────────────────────────────
    seed_results['MAML-ANIL-Uniform'] = train_maml_loso(
        loso_splits, K_SHOTS, freeze_encoder_inner=True,
        use_weighting=False, device=DEVICE,
        seed=current_seed, output_dir=OUTPUT_DIR,
        method_name='MAML-ANIL-Uniform')
    save_per_subject_metrics(seed_results['MAML-ANIL-Uniform'],
                             'MAML-ANIL-Uniform', current_seed, OUTPUT_DIR)
    save_aggregate_metrics(seed_results['MAML-ANIL-Uniform'],
                           'MAML-ANIL-Uniform', current_seed, K_SHOTS, OUTPUT_DIR)
    print("  MAML-ANIL-Uniform done")

    # ── Statistical tests ─────────────────────────────────────────────────
    comparisons = [
        ('Full-MAML',  'Full-MAML-Uniform'),
        ('MAML-ANIL',  'MAML-ANIL-Uniform'),
    ]
    stats_df = perform_statistical_tests(
        seed_results, K_SHOTS, comparisons, OUTPUT_DIR, current_seed)
    print(f"\nStatistical tests (seed {current_seed}):")
    display_cols = ['method1', 'method2', 'k', 'mean1', 'mean2',
                    'p_value', 'significant', 'effect_r', 'effect_interp']
    print(stats_df[[c for c in display_cols if c in stats_df.columns]].to_string(index=False))

    all_results_by_seed[current_seed] = seed_results

print("\n✓ Ablation 4 complete.")

## 22. Results Summary

Primary metric: **balanced accuracy** (immune to class imbalance).
Secondary: **AUC-ROC**.

Results are the mean ± std across all 26 test subjects and all 3 seeds.
Chance level = 0.5 balanced accuracy (explicit).


In [ ]:
def print_results_table(results_by_seed: dict, k_value: int = 10,
                         metric: str = 'balanced_accuracy') -> pd.DataFrame:
    """Print mean±std for a given K and metric, sorted descending."""
    all_methods = sorted({m for sr in results_by_seed.values() for m in sr})
    rows = []
    for method in all_methods:
        vals = []
        for sr in results_by_seed.values():
            if method not in sr: continue
            for sd in sr[method].get('subjects', {}).values():
                v = sd.get('k_shots', {}).get(k_value, {}).get(metric)
                if v is not None and not (isinstance(v, float) and v != v):
                    vals.append(float(v))
        if not vals: continue
        rows.append({'Method': method,
                     f'Mean': f"{np.mean(vals):.4f}",
                     f'Std':  f"{np.std(vals):.4f}",
                     'N': len(vals)})

    df = pd.DataFrame(rows).sort_values('Mean', ascending=False)
    print(f"\n{'='*55}")
    print(f"Metric: {metric.replace('_',' ').title()}  |  K={k_value}")
    print(f"Chance: 0.5000  |  Seeds={len(results_by_seed)}  |  Subjects≈26")
    print(f"{'='*55}")
    print(df.to_string(index=False))
    return df


# Print balanced accuracy and AUC for all K values
for k in Config.K_SHOTS:
    print_results_table(all_results_by_seed, k, 'balanced_accuracy')

for k in Config.K_SHOTS:
    print_results_table(all_results_by_seed, k, 'auroc')


## 23. Visualization

Three plots produced:
1. **Balanced accuracy vs K-shots** — adaptation curve per method, mean ± std
2. **AUC-ROC vs K-shots** — same
3. **Per-subject improvement heatmap** — how much SubjectConditioned improves over Supervised


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.rcParams.update({'figure.dpi': 150, 'savefig.dpi': 300,
                     'font.family': 'sans-serif', 'axes.linewidth': 1.2})

METHOD_COLORS = {
    'Supervised':          '#2ECC71',
    'Full-MAML':           '#E74C3C',
    'MAML-ANIL':           '#E67E22',
    'SubjectConditioned':  '#9B59B6',
    'Reptile':             '#3498DB',
    'Prototypical':        '#1ABC9C',
    'Matching':            '#F39C12',
    'Riemannian':          '#27AE60',
    'CovarianceAlignment': '#16A085',
    'EEGNet':              '#8E44AD',
}
METHOD_MARKERS = {
    'Supervised': 'D', 'Full-MAML': 's', 'MAML-ANIL': 'h',
    'SubjectConditioned': '*', 'Reptile': '<', 'Prototypical': '>',
    'Matching': 'X', 'Riemannian': 'd', 'CovarianceAlignment': 'P', 'EEGNet': 'o',
}


def _extract_k_metric(results_by_seed, method, k_shots, metric):
    out = {}
    for k in k_shots:
        vals = []
        for sr in results_by_seed.values():
            if method not in sr: continue
            for sd in sr[method].get('subjects', {}).values():
                v = sd.get('k_shots', {}).get(k, {}).get(metric)
                if v is not None and not (isinstance(v, float) and v != v):
                    vals.append(float(v))
        if vals:
            out[k] = (float(np.mean(vals)), float(np.std(vals)))
    return out


def plot_adaptation_curves(results_by_seed, metric='balanced_accuracy', save_path=None):
    fig, ax = plt.subplots(figsize=(12, 7))
    all_methods = sorted({m for sr in results_by_seed.values() for m in sr})

    for method in all_methods:
        kd = _extract_k_metric(results_by_seed, method, Config.K_SHOTS, metric)
        if not kd: continue
        ks = sorted(kd.keys())
        means = [kd[k][0] for k in ks]
        stds  = [kd[k][1] for k in ks]
        color  = METHOD_COLORS.get(method, '#888888')
        marker = METHOD_MARKERS.get(method, 'o')
        ax.plot(ks, means, marker=marker, linewidth=2.2, markersize=9,
                label=method, color=color, zorder=3)
        ax.fill_between(ks,
                        [m - s for m, s in zip(means, stds)],
                        [m + s for m, s in zip(means, stds)],
                        alpha=0.12, color=color, zorder=2)

    ax.axhline(0.5, color='red', linestyle=':', linewidth=1.8, alpha=0.6,
               label='Chance (0.50)', zorder=1)
    ax.set_xlabel('K-shots (support examples)', fontsize=14, fontweight='bold')
    ylabel = metric.replace('_', ' ').title()
    ax.set_ylabel(ylabel, fontsize=14, fontweight='bold')
    ax.set_title(f'{ylabel} vs K-shots\n(mean ± std over 26 subjects × 3 seeds)',
                 fontsize=15, fontweight='bold')
    ax.legend(loc='lower right', fontsize=9, framealpha=0.9)
    ax.grid(True, alpha=0.3)
    ax.set_xticks(Config.K_SHOTS)
    fig.tight_layout()
    if save_path:
        fig.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.show()
    plt.close(fig)


def plot_per_subject_heatmap(results_by_seed, method1='SubjectConditioned',
                              method2='Supervised', k=10,
                              metric='balanced_accuracy', save_path=None):
    """Heatmap of per-subject performance gain: method1 − method2."""
    m1_s = defaultdict(list)
    m2_s = defaultdict(list)
    for sr in results_by_seed.values():
        for m, store in [(method1, m1_s), (method2, m2_s)]:
            if m not in sr: continue
            for sid, sd in sr[m].get('subjects', {}).items():
                v = sd.get('k_shots', {}).get(k, {}).get(metric)
                if v is not None and not (isinstance(v, float) and v != v):
                    store[sid].append(float(v))

    common = sorted(set(m1_s) & set(m2_s))
    if not common:
        print("No common subjects found for heatmap.")
        return

    diffs = [np.mean(m1_s[s]) - np.mean(m2_s[s]) for s in common]

    fig, ax = plt.subplots(figsize=(max(14, len(common) * 0.55), 3.5))
    im = ax.imshow([diffs], cmap='RdYlGn', aspect='auto', vmin=-0.15, vmax=0.15)
    ax.set_xticks(range(len(common)))
    ax.set_xticklabels(common, rotation=45, ha='right', fontsize=9)
    ax.set_yticks([])
    ax.set_title(
        f'{metric.replace("_"," ").title()} gain: {method1} − {method2}  |  K={k}'
        f'  |  green=method1 better, red=worse',
        fontsize=12)

    for i, (sid, d) in enumerate(zip(common, diffs)):
        ax.text(i, 0, f'{d:+.3f}', ha='center', va='center',
                fontsize=7, color='black', fontweight='bold')

    plt.colorbar(im, ax=ax, fraction=0.02, pad=0.02)
    fig.tight_layout()
    if save_path:
        fig.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.show()
    plt.close(fig)


# Generate plots
for metric in ['balanced_accuracy', 'auroc']:
    plot_adaptation_curves(
        all_results_by_seed, metric=metric,
        save_path=os.path.join(Config.FIGURES_DIR, f'{metric}_vs_k.png'))

plot_per_subject_heatmap(
    all_results_by_seed, k=10,
    save_path=os.path.join(Config.FIGURES_DIR, 'per_subject_heatmap_k10.png'))

print("Figures saved to:", Config.FIGURES_DIR)


## 24. Aggregate Results CSV (All Seeds Combined)

Produces a single summary CSV averaging across all seeds for each method,
combining results into one file for external reporting.


In [ ]:
def build_combined_results_csv(results_by_seed: dict, k_shots: list,
                                output_dir: str = Config.RESULTS_DIR) -> pd.DataFrame:
    """Build a combined CSV: mean ± std per method × K across ALL seeds."""
    all_methods = sorted({m for sr in results_by_seed.values() for m in sr})
    rows = []
    for method in all_methods:
        for k in k_shots:
            vals = defaultdict(list)
            for sr in results_by_seed.values():
                if method not in sr: continue
                for sd in sr[method].get('subjects', {}).values():
                    km = sd.get('k_shots', {}).get(k, {})
                    for m in ['accuracy', 'balanced_accuracy', 'f1_score', 'auroc']:
                        v = km.get(m)
                        if v is not None and not (isinstance(v, float) and v != v):
                            vals[m].append(float(v))
            row = {'method': method, 'k_shots': k,
                   'n_seeds': len(results_by_seed)}
            for m in ['accuracy', 'balanced_accuracy', 'f1_score', 'auroc']:
                vs = vals[m]
                row[f'{m}_mean'] = float(np.mean(vs)) if vs else float('nan')
                row[f'{m}_std']  = float(np.std(vs))  if vs else float('nan')
                row[f'{m}_n']    = len(vs)
            rows.append(row)

    df = pd.DataFrame(rows).sort_values(['k_shots', 'balanced_accuracy_mean'],
                                         ascending=[True, False])
    out_path = Path(output_dir) / "combined_results_all_seeds.csv"
    df.to_csv(out_path, index=False)
    print(f"Combined results saved to: {out_path}")
    return df


combined_df = build_combined_results_csv(all_results_by_seed, Config.K_SHOTS)

# Pretty-print balanced accuracy summary
for k in Config.K_SHOTS:
    sub = combined_df[combined_df.k_shots == k][
        ['method', 'balanced_accuracy_mean', 'balanced_accuracy_std', 'auroc_mean']
    ].sort_values('balanced_accuracy_mean', ascending=False)
    print(f"\n--- K = {k} ---")
    print(sub.to_string(index=False))


## 25. Results Export

Archives the complete results directory as a zip file for download.


In [ ]:
import zipfile

zip_path = os.path.join(os.path.dirname(Config.OUTPUT_ROOT),
                        'errp_results_v2_upgraded.zip')
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
    for root, dirs, files in os.walk(Config.OUTPUT_ROOT):
        for fname in files:
            fpath   = os.path.join(root, fname)
            arcname = os.path.relpath(fpath, os.path.dirname(Config.OUTPUT_ROOT))
            zf.write(fpath, arcname)

import shutil
size_mb = os.path.getsize(zip_path) / 1e6
print(f"Results archived: {zip_path}")
print(f"Archive size    : {size_mb:.1f} MB")
